# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# # 🎬 Sparkle Movie 2 — Notebook 0 : Extraction, Enrichissement & Stockage
#
# **Objectif** : Construire la couche de données Silver prête pour l'EDA
# et la modélisation, en suivant une architecture Medallion :
#
# ```
# data/
# ├── brut/          ← Bronze : CSV originaux MovieLens (jamais modifiés)
# ├── enriched/      ← Silver : Parquet nettoyés + métadonnées TMDB
# └── gold/          ← Gold  : agrégats (produits dans le notebook EDA)
# ```
#
# **Plan du notebook** :
# 1. Configuration & imports
# 2. Chargement des CSV MovieLens → Parquet (Bronze → Silver brut)
# 3. Scraping TMDB avec cache local
# 4. Construction de la table `movies_enriched`
# 5. Validation de la Silver layer

# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ## 1. Configuration & Imports


In [1]:
import os
import json
import time
import requests
import pandas as pd
import sys
from pathlib import Path
from tqdm import tqdm

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# ── Chemins ───────────────────────────────────────────────────────────────────
DATA_BRUT     = Path("../data/brut")       # CSV MovieLens originaux
DATA_ENRICHED = Path("../data/enriched")   # Parquet Silver
DATA_ENRICHED.mkdir(parents=True, exist_ok=True)

CACHE_FILE    = DATA_ENRICHED / "tmdb_cache.json"
TMDB_API_KEY  = os.getenv("TMDB_API_KEY")  # clé dans variable d'environnement
RATE_LIMIT    = 0.05                       # 20 req/s (quota TMDB ~40/s)

# Forcer Spark à utiliser le Python du venv actuel
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

assert TMDB_API_KEY, "⚠️ Variable TMDB_API_KEY non définie !"
print("✅ Configuration OK")
print(f"  Données brutes    : {DATA_BRUT.resolve()}")
print(f"  Données enrichies : {DATA_ENRICHED.resolve()}")
print(f"Python utilisé : {sys.executable}")



✅ Configuration OK
  Données brutes    : C:\Users\romua\Documents\La_Plateforme_\Projet 7 - Sparkle Movie\sparkle-movie\data\brut
  Données enrichies : C:\Users\romua\Documents\La_Plateforme_\Projet 7 - Sparkle Movie\sparkle-movie\data\enriched
Python utilisé : c:\Users\romua\Documents\La_Plateforme_\Projet 7 - Sparkle Movie\sparkle-movie\.venv\Scripts\python.exe


In [2]:
# ── Session Spark ─────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("sparkle-movie-ingestion")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} démarré")


✅ Spark 4.1.1 démarré


# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ## 2. Chargement des CSV MovieLens → Parquet
#
# On charge les 3 fichiers MovieLens et on les persiste en **Parquet**.
# Ce format natif de Spark est compressé, typé et ~10x plus rapide
# en lecture que le CSV pour les runs suivants.
#
# Les CSV originaux dans `data/brut/` ne sont **jamais modifiés**.


In [3]:
def load_csv(filename: str, show: bool = True):
    """Charge un CSV MovieLens depuis data/brut/ et affiche un aperçu."""
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(DATA_BRUT / filename))
    )
    print(f"\n── {filename} ──")
    print(f"  Lignes   : {df.count():,}")
    print(f"  Colonnes : {df.columns}")
    df.printSchema()
    if show:
        df.show(5, truncate=60)
    return df


In [4]:
# ── Chargement ────────────────────────────────────────────────────────────────
ratings = load_csv("ratings.csv")
movies  = load_csv("movies.csv")
links   = load_csv("links.csv")



── ratings.csv ──
  Lignes   : 32,000,204
  Colonnes : ['userId', 'movieId', 'rating', 'timestamp']
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     17|   4.0|944249077|
|     1|     25|   1.0|944250228|
|     1|     29|   2.0|943230976|
|     1|     30|   5.0|944249077|
|     1|     32|   5.0|943228858|
+------+-------+------+---------+
only showing top 5 rows

── movies.csv ──
  Lignes   : 87,585
  Colonnes : ['movieId', 'title', 'genres']
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

+-------+----------------------------------+-------------------------------------------+
|movieId|                             title|                                     genres|
+-------+--------------

# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ### 2.1 Nettoyage de `movies.csv`
#
# La colonne `title` contient l'année entre parenthèses : `"Toy Story (1995)"`.
# On extrait le titre propre et l'année dans deux colonnes séparées,
# ce qui facilitera les jointures et les visualisations.


In [5]:
year_expr = F.regexp_extract(F.col("title"), r"\((\d{4})\)$", 1)

movies_clean = (
    movies
    .withColumn(
        "year_movielens",
        F.when(year_expr != "", year_expr.cast("int")).otherwise(None)
    )
    .withColumn(
        "title_clean",
        F.regexp_replace(F.col("title"), r"\s*\(\d{4}\)$", "")
    )
)


print("Aperçu après nettoyage :")
movies_clean.select("movieId", "title", "title_clean", "year_movielens", "genres").show(5, truncate=50)

# Vérification : films sans année détectée
n_no_year = movies_clean.filter(F.col("year_movielens").isNull()).count()
print(f"Films sans année détectée : {n_no_year}")


Aperçu après nettoyage :
+-------+----------------------------------+---------------------------+--------------+-------------------------------------------+
|movieId|                             title|                title_clean|year_movielens|                                     genres|
+-------+----------------------------------+---------------------------+--------------+-------------------------------------------+
|      1|                  Toy Story (1995)|                  Toy Story|          1995|Adventure|Animation|Children|Comedy|Fantasy|
|      2|                    Jumanji (1995)|                    Jumanji|          1995|                 Adventure|Children|Fantasy|
|      3|           Grumpier Old Men (1995)|           Grumpier Old Men|          1995|                             Comedy|Romance|
|      4|          Waiting to Exhale (1995)|          Waiting to Exhale|          1995|                       Comedy|Drama|Romance|
|      5|Father of the Bride Part II (1995)|Father 

# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ### 2.2 Nettoyage de `ratings.csv`
#
# Vérification des doublons (userId, movieId) et des valeurs hors plage.
# Les notes valides sont comprises entre 0.5 et 5.0 par pas de 0.5.


In [6]:
# Doublons
n_before = ratings.count()
ratings_clean = ratings.dropDuplicates(["userId", "movieId"])
n_after  = ratings_clean.count()
print(f"Doublons supprimés : {n_before - n_after:,}")

# Notes hors plage
n_invalid = ratings_clean.filter(
    (F.col("rating") < 0.5) | (F.col("rating") > 5.0)
).count()
print(f"Notes hors plage [0.5–5.0] : {n_invalid:,}")

# Valeurs nulles
for col in ratings_clean.columns:
    n_null = ratings_clean.filter(F.col(col).isNull()).count()
    if n_null > 0:
        print(f"  ⚠️  {col} : {n_null:,} nulls")
print("✅ Ratings propres")


Doublons supprimés : 0
Notes hors plage [0.5–5.0] : 0
✅ Ratings propres


# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ### 2.3 Sauvegarde en Parquet


In [7]:
ratings_clean.write.mode("overwrite").parquet(str(DATA_ENRICHED / "ratings.parquet"))
movies_clean.write.mode("overwrite").parquet(str(DATA_ENRICHED / "movies.parquet"))
links.write.mode("overwrite").parquet(str(DATA_ENRICHED / "links.parquet"))

print("✅ Parquet sauvegardés :")
print(f"  {DATA_ENRICHED}/ratings.parquet")
print(f"  {DATA_ENRICHED}/movies.parquet")
print(f"  {DATA_ENRICHED}/links.parquet")


✅ Parquet sauvegardés :
  ..\data\enriched/ratings.parquet
  ..\data\enriched/movies.parquet
  ..\data\enriched/links.parquet


# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ## 3. Scraping TMDB
#
# On enrichit chaque film avec les métadonnées TMDB via son `tmdbId`
# (disponible dans `links.csv`).
#
# **Stratégie** :
# - Un **cache JSON local** évite de re-scraper si le script est interrompu
# - Un seul appel par film grâce à `append_to_response`
# - Gestion du rate limiting (429) et des films introuvables (404)


In [ ]:
# ── Fonctions de cache ────────────────────────────────────────────────────────
def save_cache(cache: dict):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)

def load_cache() -> dict:
    if CACHE_FILE.exists():
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


In [9]:
# ── Extraction d'un film ──────────────────────────────────────────────────────
def fetch_tmdb(tmdb_id: int, cache: dict) -> dict | None:
    """
    Appelle l'API TMDB pour un film donné.
    Retourne None si le film n'existe pas (404) ou en cas d'erreur.
    Utilise le cache pour éviter les appels redondants.
    """
    key = str(tmdb_id)
    if key in cache:
        return cache[key]   # déjà scrappé → skip

    url    = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {
        "api_key": TMDB_API_KEY,
        "language": "fr-FR",
        "append_to_response": "credits,keywords,external_ids"
    }

    try:
        r = requests.get(url, params=params, timeout=10)

        if r.status_code == 404:
            cache[key] = None
            return None

        if r.status_code == 429:        # rate limit dépassé
            print("  ⏳ Rate limit atteint, pause 10s...")
            time.sleep(10)
            return fetch_tmdb(tmdb_id, cache)

        r.raise_for_status()
        d = r.json()

        crew     = d.get("credits", {}).get("crew", [])
        cast     = d.get("credits", {}).get("cast", [])
        keywords = d.get("keywords", {}).get("keywords", [])

        record = {
            # ── Identifiants ──────────────────────────────────────────────
            "tmdbId":          tmdb_id,
            "imdb_id":         d.get("external_ids", {}).get("imdb_id"),
            # ── Métadonnées ───────────────────────────────────────────────
            "title_fr":        d.get("title"),
            "original_title":  d.get("original_title"),
            "overview":        d.get("overview"),
            "release_date":    d.get("release_date"),
            "runtime":         d.get("runtime"),
            "status":          d.get("status"),
            "original_lang":   d.get("original_language"),
            "tagline":         d.get("tagline"),
            # ── Financier ─────────────────────────────────────────────────
            "budget":          d.get("budget"),
            "revenue":         d.get("revenue"),
            # ── Popularité ────────────────────────────────────────────────
            "vote_average":    d.get("vote_average"),
            "vote_count":      d.get("vote_count"),
            "popularity":      d.get("popularity"),
            # ── Listes (séparées par | pour Spark split()) ────────────────
            "genres_tmdb":     "|".join([g["name"] for g in d.get("genres", [])]),
            "keywords":        "|".join([k["name"] for k in keywords[:20]]),
            "directors":       "|".join([c["name"] for c in crew if c["job"] == "Director"]),
            "cast_top5":       "|".join([c["name"] for c in cast[:5]]),
            "studios":         "|".join([p["name"] for p in d.get("production_companies", [])]),
            # ── Images ────────────────────────────────────────────────────
            "poster_path":     d.get("poster_path"),
        }

        cache[key] = record
        time.sleep(RATE_LIMIT)
        return record

    except Exception as e:
        print(f"  ⚠️  Erreur tmdb_id={tmdb_id}: {e}")
        return None


# Lancement du scraping asynchrone avec `aiohttp` et `asyncio` pour maximiser le débit dans `scripts/scraping_tmdb.py`.

# Supprimer le cache existant pour repartir à zéro

In [21]:
import json
from pathlib import Path

CACHE_FILE = Path("../data/enriched/tmdb_cache.json")

# Charger le cache
with open(CACHE_FILE, "r", encoding="utf-8") as f:
    cache = json.load(f)

# Compter avant
n_before  = len(cache)
n_none    = sum(1 for v in cache.values() if v is None)
n_records = sum(1 for v in cache.values() if v is not None)

print(f"Entrées totales  : {n_before:,}")
print(f"Entrées valides  : {n_records:,}")
print(f"Entrées None     : {n_none:,}  ← ces IDs seront retentés")

# Supprimer uniquement les None (garder les films valides)
cache_clean = {k: v for k, v in cache.items() if v is not None}

# Sauvegarder
with open(CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump(cache_clean, f, ensure_ascii=False)

print(f"\n✅ Cache nettoyé : {len(cache_clean):,} entrées conservées")
print(f"   {n_none:,} None supprimés → seront retentés via fallback /tv/")


Entrées totales  : 87,425
Entrées valides  : 86,672
Entrées None     : 753  ← ces IDs seront retentés

✅ Cache nettoyé : 86,672 entrées conservées
   753 None supprimés → seront retentés via fallback /tv/


In [25]:
# Chargement des fichiers (déjà en mémoire si tu as run les cellules précédentes,
# sinon on les recharge depuis les Parquet)
links_spark  = spark.read.parquet(str(DATA_ENRICHED / "links.parquet"))
tmdb_spark   = spark.read.parquet(str(DATA_ENRICHED / "tmdb_metadata.parquet"))
movies_spark = spark.read.parquet(str(DATA_ENRICHED / "movies.parquet"))

# Tous les tmdbIds présents dans links
all_ids = links_spark.select("tmdbId").dropna().distinct()

# tmdbIds effectivement récupérés
found_ids = tmdb_spark.select("tmdbId").distinct()

# Anti-jointure → IDs dans links mais absents du scraping
missing_df = (
    all_ids
    .join(found_ids, on="tmdbId", how="left_anti")          # ce qui manque
    .join(links_spark.select("movieId", "tmdbId"),           # récupère movieId
          on="tmdbId", how="left")
    .join(movies_spark.select("movieId", "title"),           # récupère le titre
          on="movieId", how="left")
    .select("movieId", "tmdbId", "title")
    .orderBy("tmdbId")
)

n_total   = all_ids.count()
n_found   = found_ids.count()
n_missing = missing_df.count()

print(f"tmdbIds dans links.csv     : {n_total:,}")
print(f"Films scrappés             : {n_found:,}")
print(f"Films manquants            : {n_missing:,} ({n_missing/n_total:.1%})")
print(f"\n→ Ces films manquants sont utilisables pour ALS/KNN")
print(f"  mais exclus de la recommandation basée contenu (TF-IDF)")

print("\nAperçu des films manquants :")
missing_df.show(20, truncate=60)

# Sauvegarde pour investigation
missing_df.toPandas().to_csv(
    str(DATA_ENRICHED / "missing_tmdb.csv"),
    index=False,
    encoding="utf-8"
)
print("💾 Liste complète → data/enriched/missing_tmdb.csv")


tmdbIds dans links.csv     : 87,425
Films scrappés             : 87,042
Films manquants            : 383 (0.4%)

→ Ces films manquants sont utilisables pour ALS/KNN
  mais exclus de la recommandation basée contenu (TF-IDF)

Aperçu des films manquants :
+-------+------+------------------------------------------------------------+
|movieId|tmdbId|                                                       title|
+-------+------+------------------------------------------------------------+
|    260|    11|                   Star Wars: Episode IV - A New Hope (1977)|
|  45722|    58|           Pirates of the Caribbean: Dead Man's Chest (2006)|
|  34048|    74|                                    War of the Worlds (2005)|
|  58559|   155|                                     Dark Knight, The (2008)|
|   2114|   227|                                       Outsiders, The (1983)|
|   1084|   475|                                     Bonnie and Clyde (1967)|
|   4848|  1018|                             

# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ## 4. Construction de `movies_enriched`
#
# On assemble la table Silver finale en joignant :
# - `movies.parquet`       (MovieLens : movieId, title, genres)
# - `links.parquet`        (pont : movieId ↔ tmdbId)
# - `tmdb_metadata.parquet`(TMDB : overview, keywords, budget, etc.)
#
# ```
# movies ──(movieId)──▶ links ──(tmdbId)──▶ tmdb_metadata
# ```


In [27]:
import pyspark.sql.functions as F

movies_p  = spark.read.parquet(str(DATA_ENRICHED / "movies.parquet"))
links_p   = spark.read.parquet(str(DATA_ENRICHED / "links.parquet"))
tmdb_p    = spark.read.parquet(str(DATA_ENRICHED / "tmdb_metadata.parquet"))

# Étape 1 : movies + links → on récupère tmdbId et imdbId
movies_linked = movies_p.join(
    links_p.select("movieId", "tmdbId", "imdbId"),
    on="movieId",
    how="left"
)

# Étape 2 : + métadonnées TMDB
movies_enriched = movies_linked.join(
    tmdb_p.drop("imdb_id"),   # ← imdb_id déjà dans links, évite la colonne dupliquée
    on="tmdbId",
    how="left"
)

# Étape 3 : dédoublonnage sur movieId (clé métier principale)
movies_enriched = movies_enriched.dropDuplicates(["movieId"])

# Sauvegarde
movies_enriched.write.mode("overwrite") \
    .parquet(str(DATA_ENRICHED / "movies_enriched.parquet"))

print(f"✅ movies_enriched.parquet sauvegardé")
print(f"   Lignes   : {movies_enriched.count():,}")
print(f"   Colonnes ({len(movies_enriched.columns)}) : {movies_enriched.columns}")


✅ movies_enriched.parquet sauvegardé
   Lignes   : 87,585
   Colonnes (44) : ['tmdbId', 'movieId', 'title', 'genres', 'year_movielens', 'title_clean', 'imdbId', 'media_type', 'title_fr', 'original_title', 'tagline', 'overview', 'keywords', 'release_date', 'runtime', 'status', 'adult', 'original_lang', 'origin_countries', 'prod_countries', 'spoken_languages', 'budget', 'revenue', 'vote_average', 'vote_count', 'popularity', 'genres_tmdb', 'directors', 'writers', 'composers', 'producers', 'cinematographer', 'cast_top10', 'cast_characters', 'studios', 'collection', 'collection_id', 'certification_us', 'certification_fr', 'similar_ids', 'recommended_ids', 'poster_path', 'backdrop_path', 'real_tmdb_id']


# ── 📝 MARKDOWN ──────────────────────────────────────────────────────────────
# ## 5. Validation de la Silver Layer
#
# On vérifie la qualité de la table enrichie avant de passer à l'EDA.


In [28]:
# ── Taux de couverture TMDB ───────────────────────────────────────────────────
total      = movies_enriched.count()
with_tmdb  = movies_enriched.filter(F.col("overview").isNotNull()).count()
without    = total - with_tmdb

print(f"Films totaux            : {total:,}")
print(f"Films enrichis (TMDB)   : {with_tmdb:,} ({with_tmdb/total:.1%})")
print(f"Films sans enrichissement: {without:,} ({without/total:.1%})")
print("→ Les films sans données TMDB sont souvent des films très anciens")
print("  ou des entrées sans correspondance dans links.csv.")


Films totaux            : 87,585
Films enrichis (TMDB)   : 87,078 (99.4%)
Films sans enrichissement: 507 (0.6%)
→ Les films sans données TMDB sont souvent des films très anciens
  ou des entrées sans correspondance dans links.csv.


In [30]:
# ── Aperçu final ──────────────────────────────────────────────────────────────
movies_enriched.select(
    "movieId", "title_clean", "year_movielens",
    "genres",        # MovieLens (EN)
    "genres_tmdb",   # TMDB (FR)
    "overview", "keywords",
    "directors", "cast_top10",   # ← cast_top5 → cast_top10
    "vote_average", "budget", "revenue"
).show(5, truncate=55)


+-------+---------------------------+--------------+----------------------------+---------------------------+-------------------------------------------------------+-------------------------------------------------------+-------------------------------------------------------+-------------------------------------------------------+------------+---------+-----------+
|movieId|                title_clean|year_movielens|                      genres|                genres_tmdb|                                               overview|                                               keywords|                                              directors|                                             cast_top10|vote_average|   budget|    revenue|
+-------+---------------------------+--------------+----------------------------+---------------------------+-------------------------------------------------------+-------------------------------------------------------+-----------------------------------------

In [33]:
import json
from pathlib import Path

DATA_ENRICHED = Path("../data/enriched")
CACHE_FILE    = DATA_ENRICHED / "tmdb_cache.json"

# ── Récapitulatif des fichiers créés ──────────────────────────────────────────
print("📁 Fichiers Silver créés dans data/enriched/ :")
for f in sorted(DATA_ENRICHED.glob("*.parquet")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  ✅ {f.name:<40} ({size_mb:.1f} MB)")

# Lecture directe du cache sans importer le script
with open(CACHE_FILE, "r", encoding="utf-8") as f:
    cache = json.load(f)

n_total  = len(cache)
n_found  = sum(1 for v in cache.values() if v is not None)
n_missing = sum(1 for v in cache.values() if v is None)

print(f"\n  📄 tmdb_cache.json")
print(f"     Entrées totales  : {n_total:,}")
print(f"     Films trouvés    : {n_found:,}")
print(f"     Vraiment absents : {n_missing:,} ({n_missing/n_total:.1%})")
print("\n→ Prêt pour le Notebook 1 : EDA 🚀")


📁 Fichiers Silver créés dans data/enriched/ :
  ✅ links.parquet                            (0.0 MB)
  ✅ movies.parquet                           (0.0 MB)
  ✅ movies_enriched.parquet                  (0.0 MB)
  ✅ ratings.parquet                          (0.0 MB)
  ✅ tmdb_metadata.parquet                    (52.6 MB)

  📄 tmdb_cache.json
     Entrées totales  : 87,350
     Films trouvés    : 87,042
     Vraiment absents : 308 (0.4%)

→ Prêt pour le Notebook 1 : EDA 🚀
